# 2. The $\sigma(Wx + b)$ and $\sigma(W * x)$ nonlinearities

Notebook 01 handed you 384 numbers per image and left the obvious question open: **where
did those numbers come from?**

They came from about a dozen stacked applications of the two lines this notebook is about.
A layer is a linear operator, a bias, and a nonlinearity — and that is true whether the
linear operator is a dense matrix or a convolution. Courses usually teach those as two
different animals, in two different chapters. They are the same construction, and teaching
them together is both shorter and more honest.

The part that is *not* optional is the nonlinearity $\sigma$. That is what this notebook is
really about. §2.1–§2.4 show that without it, any stack of layers is one linear function.
§2.5 shows what it adds: bends, and with enough bends any curve. §2.6 compares the
activations you will meet in practice.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn

from notebooktester import param
from plotting import plot_activation_fits, plot_relu_buildup, plot_relu_sums

torch.manual_seed(0)

## 2.1 A stack of linear layers is one linear layer

Five `nn.Linear` layers, no activations between them. It looks like a deep network. Ask
what it can compute.

Each layer does $x \mapsto W_i x + b_i$. Write out two of them:

$$
W_2(W_1x + b_1) + b_2 \;=\; \underbrace{W_2W_1}_{W'}\,x \;+\; \underbrace{W_2b_1 + b_2}_{b'}
$$

A matrix times a matrix is a matrix, and a matrix times a vector plus a vector is a vector.
So two linear layers are one linear layer with weight $W'$ and bias $b'$, and repeating the
same step folds five layers into one as well.

You can build that single layer by multiplying the weights out by hand. The code below
leaves the biases out to keep it short, so the five layers are $W_5 W_4 W_3 W_2 W_1 x$;
§2.4 does the full version, biases included, on a trained network.

In [ ]:
WIDTH = 32
layers = [nn.Linear(WIDTH, WIDTH, bias=False) for _ in range(5)]
stack = nn.Sequential(*layers)

# collapse the five weight matrices into one.
# forward order is layers[0] -> ... -> layers[4], so the matrix product runs backwards
W = layers[-1].weight
for layer in reversed(layers[:-1]):
    W = W @ layer.weight

x = torch.rand(8, WIDTH)

with torch.no_grad():
    deep, flat = stack(x), x @ W.T

print("five layers :", tuple(deep.shape), " parameters:", sum(p.numel() for p in stack.parameters()))
print("one matrix  :", tuple(flat.shape), " parameters:", W.numel())
print()
print(f"max abs difference: {(deep - flat).abs().max():.2e}")
assert torch.allclose(deep, flat, atol=1e-5)
print("identical ✓")

5120 parameters doing the work of 1024. Not *approximately* the same function — the same
function, to floating-point precision.

This is worth being blunt about: **without a nonlinearity, depth buys you exactly
nothing.** Not "less than you hoped". Nothing. Every extra layer is a slower way to compute
a linear map you could have written down as one matrix.

## 2.2 The same is true of convolutions

A convolution is a linear operator too. It has two constraints a dense layer does not:

- **weight sharing** — the same small kernel is applied at every position, instead of a
  fresh weight per input-output pair;
- **locality** — an output only depends on a small window of the input.

Those constraints are what make it *useful* on images: they encode the assumption that a
pattern means the same thing wherever it appears, and they cut the parameter count by
orders of magnitude. But they constrain *which* linear maps the layer can express. They do
not make it a different kind of operation.

The test is the same one we just ran. If a conv layer is linear, two stacked convs with no
activation between them should collapse into a single conv — a wider one, since the
receptive fields compose.

In [ ]:
signal = torch.rand(1, 1, 20)                   # (B, C, L) -- channels second, see tensor-ladder.md

c1 = nn.Conv1d(1, 1, kernel_size=3, bias=False)
c2 = nn.Conv1d(1, 1, kernel_size=3, bias=False)

# the composed kernel: every way of picking one tap from each kernel
w1, w2 = c1.weight.flatten(), c2.weight.flatten()
composed = torch.zeros(len(w1) + len(w2) - 1)
for i, a in enumerate(w1):
    for j, b in enumerate(w2):
        composed[i + j] += a * b

single = nn.Conv1d(1, 1, kernel_size=5, bias=False)
with torch.no_grad():
    single.weight.copy_(composed.view(1, 1, -1))

with torch.no_grad():
    two, one = c2(c1(signal)), single(signal)

print("two convs, k=3 each :", tuple(two.shape), " kernel taps:", w1.numel() + w2.numel())
print("one conv,  k=5      :", tuple(one.shape), " kernel taps:", composed.numel())
print()
print(f"max abs difference: {(two - one).abs().max():.2e}")
assert torch.allclose(two, one, atol=1e-6)
print("identical ✓")

Same story, same verdict. Two 3-taps become one 5-tap: the receptive field grew, the
expressible function did not.

Note what the collapse *did* change — the shape. Each `kernel_size=3` with no padding costs
two positions, so 20 → 18 → 16, and the single 5-tap goes 20 → 16 directly. Stride and
padding change these numbers; they do not change the argument. Predicting output shapes
before running is exercise E1.

So we have two flavours of the same thing:

| | linear operator | constraint | shape it wants |
|---|---|---|---|
| **dense** | $Wx$, a full matrix | none — every input touches every output | `(..., D)` |
| **conv** | $W * x$, a shared kernel | weight sharing + locality | `(B, C, L)` or `(B, C, H, W)` |

and neither of them can express anything a single layer could not, no matter how many you
stack. Which brings us to the ingredient that changes that.

In [ ]:
# one picture, nothing more: which of the 20 input positions feed a single output
# position of the composed 5-tap kernel (output index 5 -> inputs 5..9)
receptive_field = range(5, 10)
colors = ["tab:orange" if i in receptive_field else "lightgray" for i in range(20)]

fig, ax = plt.subplots(figsize=(9, 1))
ax.bar(range(20), [1] * 20, color=colors, edgecolor="white", width=1.0)
ax.set_yticks([])
ax.set_xticks(range(20))
ax.set_title("one output position of the 5-tap kernel depends on these 5 input positions")
plt.tight_layout()
plt.show()

## 2.3 What the nonlinearity is for

Put a `GELU` between the layers of §2.1 and re-run the collapse. The matrix product is
still a perfectly good matrix — it just no longer describes what the network does.

In [ ]:
gelu_stack = nn.Sequential(
    layers[0], nn.GELU(),
    layers[1], nn.GELU(),
    layers[2], nn.GELU(),
    layers[3], nn.GELU(),
    layers[4],
)

x = torch.rand(8, WIDTH)
with torch.no_grad():
    bent, flat = gelu_stack(x), x @ W.T

print(f"max abs difference from the collapsed matrix: {(bent - flat).abs().max():.4f}")
print("still identical?", torch.allclose(bent, flat, atol=1e-5))

That was a numeric argument: two functions, compared by how close their outputs land.
Here is a visual one, on a task simple enough to see completely.

Four clusters sit at the corners of a square, coloured so that **diagonal corners share a
class** — this is XOR, the textbook example of "not linearly separable," old but exactly
the point. No single straight line divides the plane into "class 0" and "class 1"
correctly, whatever its angle or offset. Watch what a `Linear` layer can do with that
compared to a GELU network.

In [ ]:
torch.manual_seed(0)
n = 300
corners = torch.randint(0, 2, (n, 2)).float() * 2 - 1        # each coord is +-1
points = corners + 0.25 * torch.randn(n, 2)                  # a cloud around each corner
xor_labels = (corners[:, 0] * corners[:, 1] < 0).long()       # opposite-sign corners -> class 1


def fit_classifier(model, steps=param(400, test=20), lr=1e-2):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(steps):
        opt.zero_grad()
        loss_fn(model(points), xor_labels).backward()
        opt.step()
    return model


torch.manual_seed(0)
linear_clf = fit_classifier(nn.Linear(2, 2))
torch.manual_seed(0)
nonlin_clf = fit_classifier(nn.Sequential(nn.Linear(2, 32), nn.GELU(), nn.Linear(32, 2)))

gx, gy = torch.meshgrid(torch.linspace(-1.8, 1.8, 150), torch.linspace(-1.8, 1.8, 150), indexing="xy")
grid = torch.stack([gx.flatten(), gy.flatten()], dim=1)
colors = ["tab:blue" if label == 0 else "tab:orange" for label in xor_labels]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, model, title in [(axes[0], linear_clf, "linear: Wx + b"), (axes[1], nonlin_clf, "nonlinear: GELU MLP")]:
    with torch.no_grad():
        boundary = model(grid).argmax(1).reshape(gx.shape)
        accuracy = (model(points).argmax(1) == xor_labels).float().mean()
    ax.contourf(gx, gy, boundary, levels=[-0.5, 0.5, 1.5], alpha=0.3, colors=["tab:blue", "tab:orange"])
    ax.scatter(points[:, 0], points[:, 1], c=colors, edgecolor="k", s=20)
    ax.set_title(f"{title}\naccuracy {accuracy:.2f}")
plt.tight_layout()
plt.show()

The linear model's boundary is a straight line (or close to it — §2.1 proved that is all a
stack of `Linear`s can ever draw), and a straight line gets roughly half of XOR right
regardless of how long you train it. That is not a training failure, it is a
representational one: no setting of `W` and `b` fixes it. The GELU model bends the boundary
around the diagonal split and gets it essentially right.

## 2.4 The sine test: same layers, with and without an activation

Same idea as a regression, where you can see the whole function the network computes.
The target is $y = \sin(3x)$ on $[-3, 3]$, with some noise on the training targets.

Two networks of the same shape, $1 \to 64 \to 64 \to 1$, trained the same way. The only
difference: the second one has a `GELU` after each hidden layer.

In [ ]:
STEPS = param(2000, test=50)

xs = torch.linspace(-3, 3, 200).unsqueeze(1)
true_ys = torch.sin(3 * xs)

torch.manual_seed(1)
ys = true_ys + 0.15 * torch.randn_like(true_ys)   # noisy targets -- something to actually fit


def fit(model, steps=STEPS, lr=1e-2):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        loss = ((model(xs) - ys) ** 2).mean()
        loss.backward()
        opt.step()
    return model, float(loss.detach())


torch.manual_seed(0)
linear, linear_loss = fit(nn.Sequential(nn.Linear(1, 64), nn.Linear(64, 64), nn.Linear(64, 1)))
torch.manual_seed(0)
nonlin, nonlin_loss = fit(nn.Sequential(nn.Linear(1, 64), nn.GELU(), nn.Linear(64, 64), nn.GELU(), nn.Linear(64, 1)))

print(f"three linear layers, no activation : final MSE {linear_loss:.3f} (against noisy targets)")
print(f"the same three, with GELU          : final MSE {nonlin_loss:.3f} (against noisy targets)")
print(f"\nthe nonlinear stack is {linear_loss / max(nonlin_loss, 1e-12):.1f}x better on the same data")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
panels = [
    (axes[0], linear, "Linear → Linear → Linear", linear_loss),
    (axes[1], nonlin, "Linear → GELU → Linear → GELU → Linear", nonlin_loss),
]
with torch.no_grad():
    for ax, model, title, loss in panels:
        ax.scatter(xs, ys, color="gray", s=8, alpha=0.5, label="noisy training targets")
        ax.plot(xs, true_ys, color="k", lw=1.5, ls="--", label="true function sin(3x)")
        ax.plot(xs, model(xs), color="tab:red", lw=3, label="what the network draws")
        ax.set_title(f"{title}\nMSE {loss:.3f}")
        ax.set_xlabel("x")
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

The left panel is the point of this whole notebook. More than four thousand parameters,
trained for thousands of steps, and the best it can draw is a straight line. That is not a
training failure: by §2.1 a stack of `Linear` layers *is* one `Linear` layer, and a
`Linear(1, 1)` is a line $y = wx + b$.

You can check that on the trained network. Write out its three layers:

$$
W_3\big(W_2(W_1x + b_1) + b_2\big) + b_3
\;=\; \underbrace{W_3W_2W_1}_{W'}\,x \;+\; \underbrace{W_3W_2b_1 + W_3b_2 + b_3}_{b'}
$$

The weights have shapes `(64, 1)`, `(64, 64)` and `(1, 64)`, so $W'$ has shape `(1, 1)`
and $b'$ has shape `(1,)`: two numbers. Compute them, load them into an `nn.Linear(1, 1)`,
and compare it with the three-layer stack. Then compare both with the best straight line
through the data, which least squares gives you directly, without any training.

In [ ]:
l1, l2, l3 = linear                   # the three nn.Linear layers of the trained linear stack

with torch.no_grad():
    W_prime = l3.weight @ l2.weight @ l1.weight                      # (1,64) @ (64,64) @ (64,1) -> (1,1)
    b_prime = l3.weight @ (l2.weight @ l1.bias + l2.bias) + l3.bias  # -> (1,)

    one = nn.Linear(1, 1)
    one.weight.copy_(W_prime)
    one.bias.copy_(b_prime)

    print(f"three layers: {sum(p.numel() for p in linear.parameters()):5d} parameters")
    print(f"one layer   : {sum(p.numel() for p in one.parameters()):5d} parameters"
          f"   y = {W_prime.item():.4f} x {b_prime.item():+.4f}")
    print(f"max abs difference between them: {(linear(xs) - one(xs)).abs().max():.2e}")
    assert torch.allclose(linear(xs), one(xs), atol=1e-5)

    # the best straight line through the data, solved in one step by least squares
    X = torch.cat([xs, torch.ones_like(xs)], dim=1)                  # columns: x, 1
    w_ls, b_ls = torch.linalg.lstsq(X, ys).solution.flatten()
    print(f"\nleast-squares line:          y = {w_ls:.4f} x {b_ls:+.4f}")

The three trained layers are one line, and it is the same line least squares finds in a
single step. Training did its job perfectly; the model just has nothing better than a line
to offer. More layers would not change that, because they collapse away too. The only edit
that changes what the network can draw is the activation.

## 2.5 How a nonlinearity builds a curve

Why does one activation make that much difference? The clearest picture of it is in
chapter 3 of the course book, *Understanding Deep Learning* (Prince, 2023). It uses ReLU,
$\mathrm{ReLU}(z) = \max(0, z)$, because its kinks are easy to see. Figure 3.3 (p. 28)
takes a network with one input, three hidden units and one output, and follows the
computation step by step:

![Understanding Deep Learning, Figure 3.3](../assets/udl/udl_fig3.3_shallow_buildup.png)

*Figure 3.3 from S.J.D. Prince, [Understanding Deep Learning](https://udlbook.github.io/udlbook/)
(MIT Press, 2023), CC BY-NC-ND 4.0.*

Read it from top to bottom:

1. **a–c** Each hidden unit computes a line $\theta_{d0} + \theta_{d1}x$. Together these are
   the first `nn.Linear(1, 3)`: three weights, three biases.
2. **d–f** ReLU clips each line at zero. Each line becomes a *hinge*: flat on one side,
   sloped on the other, with a kink where the line crossed zero.
3. **g–i** The second `nn.Linear(3, 1)` scales each hinge by its weight $\phi_d$.
4. **j** The hinges are added up, plus a bias $\phi_0$. Every kink of a hinge becomes a
   kink of the sum, so three units give a curve of up to four straight pieces.

Without step 2 the three lines would add up to one line, which is §2.1 again. The ReLU is
what lets each unit put its own bend in its own place.

Here is the same computation in PyTorch. The parameters come from the book's Notebook 3.1,
so the curve differs from the figure, but each row of panels is one line of tensor code.

In [ ]:
# parameters from UDL Notebook 3.1 "Shallow networks I" (© 2023 Simon Prince, MIT licence)
theta_0 = [0.3, -1.0, -0.5]    # intercept of each hidden unit's line
theta_1 = [-1.0, 2.0, 0.65]    # slope of each hidden unit's line
phi = [2.0, -1.0, 7.0]         # output weight of each hidden unit
phi_0 = -0.3                   # output bias

shallow = nn.Sequential(nn.Linear(1, 3), nn.ReLU(), nn.Linear(3, 1))
with torch.no_grad():
    shallow[0].weight.copy_(torch.tensor(theta_1).unsqueeze(1))   # (3, 1): one row per unit
    shallow[0].bias.copy_(torch.tensor(theta_0))
    shallow[2].weight.copy_(torch.tensor(phi).unsqueeze(0))       # (1, 3)
    shallow[2].bias.fill_(phi_0)

x01 = torch.linspace(0, 1, 200).unsqueeze(1)
with torch.no_grad():
    lines = shallow[0](x01)                                        # (200, 3)  panels a-c
    hinges = torch.relu(lines)                                     # (200, 3)  panels d-f
    weighted = hinges * shallow[2].weight                          # (200, 3)  panels g-i
    output = weighted.sum(dim=1, keepdim=True) + shallow[2].bias   # (200, 1)  panel j

assert torch.allclose(output, shallow(x01).detach())

plot_relu_buildup(x01, lines, hinges, weighted, output)
plt.show()

### More units, more pieces

Each hidden unit adds one kink, so a ReLU network with $D$ hidden units draws a curve of up
to $D + 1$ straight pieces. With enough pieces you can follow any continuous curve as
closely as you like. That is the **universal approximation theorem** (UDL §3.2; figure 3.5
on p. 30 shows it with 5, 10 and 20 pieces; below are the same three counts).

You do not have to take that on faith, because you can build such a network by hand,
without training. Pick knots $t_0 < t_1 < \dots < t_D$ on the x-axis and connect the points
$(t_j, \sin 3t_j)$ with straight segments:

- hidden unit $j$ is $\mathrm{ReLU}(x - t_j)$: weight 1, bias $-t_j$, so it switches on at
  knot $t_j$;
- its output weight is the *change in slope* at that knot, so that from $t_j$ on the curve
  heads straight for the next point;
- the output bias is the starting height $\sin 3t_0$.

In [ ]:
def relu_through(knots_x, knots_y):
    "A one-hidden-layer ReLU network that runs straight from knot to knot, built by hand."
    slopes = (knots_y[1:] - knots_y[:-1]) / (knots_x[1:] - knots_x[:-1])   # one per segment
    n_units = len(slopes)
    net = nn.Sequential(nn.Linear(1, n_units), nn.ReLU(), nn.Linear(n_units, 1))
    with torch.no_grad():
        net[0].weight.fill_(1.0)
        net[0].bias.copy_(-knots_x[:-1])                    # unit j switches on at knot j
        slope_change = torch.diff(slopes, prepend=torch.zeros(1))
        net[2].weight.copy_(slope_change.unsqueeze(0))
        net[2].bias.fill_(knots_y[0])
    return net


def weighted_units(net, x):
    "Each hidden unit's contribution to the output: ReLU(line) times its output weight."
    with torch.no_grad():
        return torch.relu(net[0](x)) * net[2].weight      # (len(x), units)


pieces, totals, titles = [], [], []
for n_units in (5, 10, 20):
    knots = torch.linspace(-3, 3, n_units + 1)
    net = relu_through(knots, torch.sin(3 * knots))
    with torch.no_grad():
        approx = net(xs)
    pieces.append(weighted_units(net, xs))
    totals.append(approx)
    titles.append(f"{n_units} units by hand\nMSE vs sin(3x): {((approx - true_ys) ** 2).mean():.4f}")

plot_relu_sums(xs, pieces, totals, true_ys, titles)
plt.show()

Top row: every hidden unit's weighted hinge. Bottom row: their sum. Five pieces cannot
follow a sine with almost three periods, ten trace every rise and fall roughly, and at
twenty the remaining error is hard to see. Nothing here was trained; the weights were
written down.

Nobody places the knots by hand in practice. Hidden unit $j$ has its kink where its line
crosses zero, at $x = -b_j / w_j$, so every gradient step on $w_j$ and $b_j$ slides that kink
along the axis and tilts its hinge. Train a 20-unit ReLU network on the same data with the
`fit` from §2.4, take it apart the same way, and put it next to the hand-built one:

In [ ]:
def kinks(net):
    "Where each hidden unit's line crosses zero: x = -b / w."
    with torch.no_grad():
        return sorted(round(k, 1) for k in (-net[0].bias / net[0].weight.flatten()).tolist())


torch.manual_seed(0)
trained = nn.Sequential(nn.Linear(1, 20), nn.ReLU(), nn.Linear(20, 1))
print("kinks before training:", kinks(trained))
trained, _ = fit(trained)
print("kinks after training: ", kinks(trained))

knots = torch.linspace(-3, 3, 21)
by_hand = relu_through(knots, torch.sin(3 * knots))

pieces, totals, titles = [], [], []
for net, name in [(by_hand, "20 units, knots placed by hand"), (trained, "20 units, trained")]:
    with torch.no_grad():
        approx = net(xs)
    pieces.append(weighted_units(net, xs))
    totals.append(approx)
    titles.append(f"{name}\nMSE vs sin(3x): {((approx - true_ys) ** 2).mean():.4f}")

plot_relu_sums(xs, pieces, totals, true_ys, titles)
plt.show()

Same number of units, a much worse fit. The printed kinks show why. PyTorch's default
initialisation put most of the kinks between −1.7 and 1.7, and a few far outside the data,
where a hinge is just a straight line across the whole plot. Training moved them only a
little: afterwards, 14 of the 20 kinks sit between −1.5 and 1.5, six sit outside $[-3, 3]$,
and none are left for the outer bumps of the sine, which the network crosses with a single
straight line.

That is the difference between what a network *can* represent and what training *finds*.
The universal approximation theorem only promises that good weights exist; it says nothing
about whether gradient descent reaches them from where it starts. The GELU network of §2.4,
with more units and a second hidden layer, did reach a good fit. How training moves the
weights is the subject of notebook 03.

## 2.6 Which activation?

Every activation in common use is a way of bending a line, and any of them gives the
network what §2.4's linear stack was missing. Below are the seven you will meet most:
the top row shows each function (solid) and its slope (dashed); the bottom row is §2.4's
sine fit with that activation in place of GELU, everything else unchanged.

In [ ]:
acts = {
    "ReLU": nn.ReLU(),
    "LeakyReLU": nn.LeakyReLU(0.1),
    "ELU": nn.ELU(),
    "GELU": nn.GELU(),
    "SiLU": nn.SiLU(),
    "Sigmoid": nn.Sigmoid(),
    "Tanh": nn.Tanh(),
}

fits = {}
for name, act in acts.items():
    torch.manual_seed(0)
    model, loss = fit(nn.Sequential(nn.Linear(1, 64), act, nn.Linear(64, 64), act, nn.Linear(64, 1)))
    with torch.no_grad():
        fits[name] = (model(xs), loss)
    print(f"{name:10s} final MSE {loss:.3f}")

plot_activation_fits(acts, fits, xs, ys)
plt.show()

All seven draw the sine. The noise on the targets has standard deviation 0.15, so a model
that learned the function and ignored the noise ends up near an MSE of $0.15^2 \approx 0.023$,
and every one of them lands there. Slightly below that means it has started fitting the
noise, which shows as small wiggles (Tanh around $x = 0$). What differs is the shape between
the data points: ReLU and LeakyReLU draw straight pieces with kinks, the others smooth
curves.

On a network this small the choice hardly matters. In deep networks it does, and the top
row shows why. The dashed slope is the factor a gradient is multiplied by on its way back
through the activation, once per layer:

- **Sigmoid / Tanh saturate.** Away from zero the slope goes to zero, so the gradient
  arriving from above gets multiplied by almost nothing. Stack a few layers of that and the
  early layers stop learning — the *vanishing gradient* problem. Sigmoid is worse: its slope
  is never above 0.25. This is the historical reason deep nets were considered untrainable.
- **ReLU** has slope exactly 1 on the positive side, which fixed that. Its failure mode is
  the other one: a unit pushed permanently negative has slope 0 forever and never
  recovers — a *dead* ReLU.
- **LeakyReLU and ELU** keep some slope on the negative side (a fixed 0.1 for LeakyReLU
  here, a smooth curve that levels off at −1 for ELU), so a unit pushed negative can still
  recover.
- **GELU and SiLU** are ReLU with the corner smoothed off, with a small dip below zero. They
  keep a small gradient alive on the negative side. GELU is the default in modern
  transformers, and it is what `vectormesh.components.NeuralNet` uses; SiLU is the building
  block of the SwiGLU layers in many recent language models.

Pick GELU or ReLU and move on. The choice between them is worth far less of your attention
than the choice of encoder in notebook 01 — and notebook 04 will put a number on that claim.

## 2.7 Reading the encoder you have been using

Close the loop. Those 384 numbers in your cache came out of a stack of exactly these two
lines. Count what is actually inside the two encoders from notebook 01.

In [ ]:
from collections import Counter

from transformers import AutoModel

for tag in ("microsoft/resnet-18", "facebook/dinov2-small"):
    model = AutoModel.from_pretrained(tag)
    counts = Counter(type(m).__name__ for _, m in model.named_modules())
    params = sum(p.numel() for p in model.parameters())

    print(f"\n{tag}   ({params / 1e6:.1f}M parameters)")
    print("  top-level:", [name for name, _ in model.named_children()])
    for kind in ("Conv2d", "Linear", "BatchNorm2d", "LayerNorm", "ReLU", "GELUActivation"):
        if counts[kind]:
            print(f"    {counts[kind]:3d}  {kind}")

Two architectures, one recipe.

**resnet-18** is 20 `Conv2d`, 20 `BatchNorm2d`, 17 `ReLU`. That is §2.2's linear operator,
a normalisation, and §2.3's nonlinearity — repeated seventeen times.

**dinov2-small** is 72 `Linear`, 25 `LayerNorm`, 12 `GELUActivation`. Same recipe, dense
instead of convolutional, twelve times. (The `Linear` count is high because each attention
block spends four of them on queries, keys, values and the output projection — lesson 5's
subject, not this one.)

And look at the top of the DINOv2 count: **one `Conv2d`**. A vision transformer is not
conv-free. It uses exactly one convolution, at the very start, with a stride equal to its
kernel size — which is a tidy way of saying "cut the image into non-overlapping patches and
project each one". §2.2's operator, used once, as a tokenizer.

Neither of them is doing anything you have not now built by hand. The difference between
your five-layer stack and DINOv2 is not the construction; it is 22 million parameters and
a very large pile of images, and *that* is the thing you downloaded rather than trained.

Which is the point notebook 01 made with a stopwatch, arrived at from the other direction:
the encoder is the expensive, valuable part, and it is made of this.

## What you now know

- A stack of linear layers, dense or convolutional, collapses into a single layer:
  $W_2(W_1x + b_1) + b_2 = W'x + b'$. Depth without a nonlinearity is a slower way to
  compute the same function. A trained three-layer linear stack is one `nn.Linear(1, 1)`,
  and it is exactly the least-squares line.
- A convolution is a linear operator with weight sharing and locality. Those constraints
  buy efficiency and an assumption about the data; they do not change what kind of
  operation it is.
- The activation is the one ingredient that makes depth mean something. A ReLU network
  is a sum of hinges: each hidden unit adds one kink, and with enough units the sum follows
  any continuous curve (the universal approximation theorem, UDL chapter 3). Training is
  what places the kinks.
- Which activation matters much less than having one. Saturation and dead units are the
  two failure modes to know; GELU/ReLU avoid the first, GELU softens the second.
- Your cached encoder is this construction, repeated a dozen times, with weights someone
  else paid to fit.

## Exercises

**E1 — predict the shape.** For `x` of shape `(1, 1, 20)`, work out the output shape of
`nn.Conv1d(1, 1, kernel_size=5)` with (a) `stride=1, padding=0`, (b) `stride=2, padding=2`.
Then run it. If you were wrong, work out which term in $\lfloor (L + 2p - k)/s \rfloor + 1$
you mis-read.

**E2 — remove one activation.** In the GELU network of §2.4, replace only the *second*
`nn.GELU()` with `nn.Identity()`. Before running it, predict whether it still fits the
sine, and say which layers collapse into one. Then do the same with only the *first* GELU
replaced, and explain why the two answers are not the same.

**E3 — how deep before it stops mattering?** Repeat §2.1 with 2, 5, 10 and 50 layers and
time the forward pass of the stack against the collapsed matrix. Depth costs you something
even when it buys nothing; how much?

**E4 — collapse any linear stack.** §2.4 multiplied out a three-layer stack by hand.
Write `collapse(stack: nn.Sequential) -> nn.Linear` that does it for any number of
`nn.Linear` layers, biases included. Rebuild §2.1's five-layer stack with `bias=True` and
assert that `collapse(stack)(x)` matches `stack(x)`.